# Quadrat-based statistics in `pointpats`

This notebook demonstrates quadrat-based methods for analyzing planar point
patterns using the current `pointpats.quadrat_statistics` implementation.

The workflow is:

1. Represent the point pattern as a NumPy array of coordinates with shape `(n, 2)`.
2. Define the **study window** (a shapely polygon or multipolygon, possibly with
   holes). If no window is supplied, the minimum bounding box (MBB) of the points
   is used.
3. Build a rectangular or hexagonal tessellation **once**, anchored to the study
   window and clipped to it. This single grid plan is reused for the observed
   pattern, every simulated realization, the per-cell contributions and plots.
4. Compute a Pearson chi-square statistic from per-quadrat counts with
   **area-proportional expected counts**.
5. Optionally, obtain a **Monte Carlo p-value** by simulating CSR realizations
   within the same study window and counting them on the same grid plan, with
   reproducibility controlled by `rng`.

Key implementation facts (relevant for interpretation):

- The grid origin is the lower-left corner of the window's bounding box; it is
  **not** recomputed from a sample's bounding box.
- Each grid cell is clipped to the window. Expected counts are proportional to the
  clipped (**exposed**) area, so holes and disjoint window parts are handled
  correctly.
- Cells with zero exposed area are dropped and therefore do **not** contribute to
  the degrees of freedom.
- A point on a shared cell boundary belongs to exactly one cell: the one with the
  smallest `cell_id`.
- Observed points outside the window are handled by the `outside` policy
  (`"exclude"` or `"raise"`) and are reported on the result object.

## 1. Quadrat statistics and CSR

Quadrat analysis partitions space into a set of non-overlapping cells and compares
the observed counts per cell to what is expected under **complete spatial randomness (CSR)**.

Let $O_i$ be the count in cell \(i\), $A_i$ its exposed area, and
$A = \sum_i A_i$ the total exposed area. Under homogeneous CSR with $N$ observed
events inside the window, the expected count in cell \(i\) is

$$
E_i = N \frac{A_i}{A}.
$$

The implemented test statistic is the Pearson chi-square statistic

$$
X^2 = \sum_{i=1}^k \frac{(O_i - E_i)^2}{E_i},
$$

summed over the $k$ cells with positive exposed area. Analytical inference compares
$X^2$ to a chi-square distribution with $(k-1)$ degrees of freedom. Simulation-based
inference (Monte Carlo) instead compares the observed $X^2$ to the sampling
distribution obtained from CSR realizations generated in the same window and
counted on the same grid.


In [ ]:
import geopandas as gpd
import libpysal

%matplotlib inline
import warnings

warnings.filterwarnings("ignore")

Import `pointpats.quadrat_statistics` and compute quadrat statistics.

The main entry point is:

```python
QStatistic(points, shape="rectangle"|"hexagon", nx=3, ny=3, lh=10,
           realizations=0, window=None, outside="exclude", rng=None)
```

- `points` is always a NumPy array of shape `(n, 2)`.
- `shape` selects the tessellation type; `nx`/`ny` (rectangle) or `lh` (hexagon)
  control the resolution.
- `window` defines the study region. If omitted, the points' MBB is used.
- `outside` controls what happens to points outside the window: `"exclude"`
  (drop and report) or `"raise"`.
- `realizations > 0` enables Monte Carlo inference.
- `rng` controls reproducibility of simulations.

The constructed grid is available as `result.plan`, with aligned
`result.cell_ids`, `result.exposed_area`, `result.area_fraction`,
`result.expected_counts`, `result.observed_counts` and `result.chi2_contrib`.


In [ ]:
from pointpats.quadrat_statistics import QStatistic

Open the point shapefile and extract coordinates.

Here we use `GeoDataFrame.get_coordinates()` to obtain an `(n, 2)` NumPy array,
which is the required input type for `QStatistic`.


In [ ]:
juv = gpd.read_file(libpysal.examples.get_path("juvenile.shp"))

In [ ]:
juv.shape

In [ ]:
xy = juv.get_coordinates()

In [ ]:
xy

In [ ]:
res_rect = QStatistic(xy)

In [ ]:
res_rect.chi2_pvalue

In [ ]:
res_rect.plot()

In [ ]:
res_hex = QStatistic(xy, shape="hexagon")

In [ ]:
res_hex.plot()

In [ ]:
res_hex.chi2_pvalue

## 3. Simulation-based inference (Monte Carlo)

When `realizations > 0`, CSR realizations are generated **within the same window**
and counted on the **same grid plan**. The origin, `cell_id`s, clipped cell
geometry and cell order are fixed by the observed pattern and the window; a
realization's own sample bounding box is never used to build a grid. Each
realization's statistic uses its own point total with the same area-proportional
expected counts. The Monte Carlo p-value uses the standard +1 correction:

$$
p = \frac{\#\{X^2_{sim} \ge X^2_{obs}\} + 1}{R + 1},
$$

where $R$ is the number of realizations.

### Reproducibility

All simulation-based results can be made reproducible by supplying `rng`:

- `None` (default): create a new `numpy.random.default_rng()`
- integer seed
- `numpy.random.Generator`
- legacy `numpy.random.RandomState` (wrapped)

The same generator is passed through the CSR simulation code, ensuring repeatable
Monte Carlo p-values.


In [ ]:
from numpy.random import default_rng

In [ ]:
rng = default_rng(42)

In [ ]:
res_hex_sim = QStatistic(xy, shape="hexagon", realizations=99, rng=rng)

In [ ]:
res_hex_sim.chi2_r_pvalue

In [ ]:
res_hex_sim.chi2_realizations

In [ ]:
res_hex_sim.chi2

## 4. Irregular windows

The grid spans the study window's bounding box and every cell is clipped to the
window. This matters when the window is irregular (for example a polygon,
a polygon with a hole, or a multipolygon):

- Expected counts are **area-proportional**: $E_i = N A_i / A$, where $A_i$ is the
  clipped (exposed) area of cell \(i\).
- Grid cells with zero exposed area (entirely outside the window, or touching it
  only along an edge) are dropped, so they do not inflate the degrees of freedom.
- A point on a shared cell boundary is assigned to exactly one cell, the one with
  the smallest `cell_id`; this rule is the same for observations and simulations.
- Points strictly outside the window (including inside a hole) are excluded from
  the counts and reported in `result.outside_points` / `result.n_outside`. Use
  `outside="raise"` to reject them instead.

### How Monte Carlo interacts with this

Because observations and realizations share the same plan, comparing the observed
statistic to simulated statistics is a valid Monte Carlo test for the statistic
being used, and the reference distribution is window-aware.


In [ ]:
from shapely.geometry import Polygon

In [ ]:
window = Polygon([[0, 0], [1, 0], [0, 1]])

In [ ]:
from pointpats.random import poisson

In [ ]:
pp = poisson(window, 100 / 1, rng=42)

In [ ]:
pp

In [ ]:
qs = QStatistic(pp, nx=5, ny=5, window=window)
len(qs.cell_ids), qs.df, qs.n_outside


In [ ]:
qs.exposed_area.sum(), window.area


In [ ]:
qs.plot()


In [ ]:
qs.chi2_pvalue


In [ ]:
qsw = QStatistic(pp, nx=5, ny=5, window=window, realizations=99, rng=42)
qsw.n_outside


In [ ]:
qsw.chi2_r_pvalue

In [ ]:
qsw.plot()

## 5. Recap

In this notebook, we:

- represented a point pattern as an `(n, 2)` NumPy coordinate array,
- built a single rectangular or hexagonal grid plan anchored to the study window
  and clipped to it,
- used a Pearson chi-square statistic with area-proportional expected counts and
  excluded zero-exposure cells from the degrees of freedom,
- counted observations and all simulated realizations on the same grid plan
  (never rebuilding it from a sample's bounding box),
- applied a deterministic, unique boundary-ownership rule, and
- reported points outside the window through an explicit policy,
- obtained either an analytical p-value or a Monte Carlo p-value from CSR
  realizations, with reproducibility via an explicit `rng`.
